### Car Price Prediction System

Name: Lakshay Saini

Registration Number: 23BAI10055

Application Number: IN26011317

Batch Number: 2B

Email ID: jattabhiraj26@gmail.com



Building a car price prediction model using supervised learning (Random Forest), saving it with pickle, and deploying it on a Flask web app running on localhost.

**Dataset**: Vehicle Dataset from CarDekho (Kaggle)

**Tech Stack:** Python, Scikit-learn, Pandas, Flask, Pickle

In [ ]:
# Cell 1: Install all required packages
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn flask pickle-mixin seaborn matplotlib

In [ ]:
# importing all the stuff we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import pickle
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# loading it into a dataframe
df = pd.read_csv("/content/car data.csv")
print("shape:", df.shape)
df.head()

shape: (301, 9)


,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [ ]:
# lets see what we're working with
print(df.shape)
print()
print(df.info())
print()
print(df.describe())
print()

# checking for null values
print("null values:\n", df.isnull().sum())
print()

# columns we have
print("columns:", df.columns.tolist())

(301, 9)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    object 
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Kms_Driven     301 non-null    int64  
 5   Fuel_Type      301 non-null    object 
 6   Seller_Type    301 non-null    object 
 7   Transmission   301 non-null    object 
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 21.3+ KB
None

              Year  Selling_Price  Present_Price     Kms_Driven       Owner
count   301.000000     301.000000     301.000000     301.000000  301.000000
mean   2013.627907       4.661296       7.628472   36947.205980    0.043189
std       2.891554       5.082812       8.644115   38886.883882    0.247915
min    2003.000000       0.10

In [ ]:
# no null values so thats nice, lets clean up the data now

# making a new column for car age instead of year (makes more sense)
df['Car_Age'] = 2025 - df['Year']

# dropping car name and year cuz we dont need them anymore
df.drop(['Car_Name', 'Year'], axis=1, inplace=True)

# encoding the categorical columns manually
df['Fuel_Type'] = df['Fuel_Type'].map({'Petrol': 0, 'Diesel': 1, 'CNG': 2})
df['Seller_Type'] = df['Seller_Type'].map({'Dealer': 0, 'Individual': 1})
df['Transmission'] = df['Transmission'].map({'Manual': 0, 'Automatic': 1})

print(df.head())
print()
print(df.dtypes)

   Selling_Price  Present_Price  Kms_Driven  Fuel_Type  Seller_Type  \
0           3.35           5.59       27000          0            0   
1           4.75           9.54       43000          1            0   
2           7.25           9.85        6900          0            0   
3           2.85           4.15        5200          0            0   
4           4.60           6.87       42450          1            0   

   Transmission  Owner  Car_Age  
0             0      0       11  
1             0      0       12  
2             0      0        8  
3             0      0       14  
4             0      0       11  

Selling_Price    float64
Present_Price    float64
Kms_Driven         int64
Fuel_Type          int64
Seller_Type        int64
Transmission       int64
Owner              int64
Car_Age            int64
dtype: object


In [ ]:
# splitting into X and y
X = df.drop('Selling_Price', axis=1)
y = df['Selling_Price']

# 80-20 split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# using random forest cuz it usually gives better results than linear regression
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# checking how well it does
y_pred = model.predict(X_test)
print("R2 score:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

R2 score: 0.9582648830157845
RMSE: 0.9805064072988497


In [ ]:
# saving the model so we can use it in flask later
pickle.dump(model, open('car_price_model.pkl', 'wb'))

# just loading it back to make sure it works
loaded_model = pickle.load(open('car_price_model.pkl', 'rb'))
print("pickle saved and loaded successfully")
print("quick test prediction:", loaded_model.predict([X_test.iloc[0]]))

pickle saved and loaded successfully
quick test prediction: [0.4451]


In [ ]:
%%writefile app.py

from flask import Flask, render_template, request
import pickle
import numpy as np

app = Flask(__name__)

# loading our saved model
model = pickle.load(open('car_price_model.pkl', 'rb'))

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    # grabbing all the values from the form
    present_price = float(request.form['present_price'])
    kms_driven = int(request.form['kms_driven'])
    fuel_type = int(request.form['fuel_type'])
    seller_type = int(request.form['seller_type'])
    transmission = int(request.form['transmission'])
    owner = int(request.form['owner'])
    car_age = int(request.form['car_age'])

    # putting it all together for prediction
    features = np.array([[present_price, kms_driven, fuel_type, seller_type, transmission, owner, car_age]])
    prediction = model.predict(features)[0]

    return render_template('index.html', prediction_text=f'Estimated Car Price: ₹ {prediction:.2f} Lakhs')

if __name__ == '__main__':
    app.run(debug=True)

Writing app.py


In [ ]:
import os

html_content = '''
<!DOCTYPE html>
<html>
<head>
    <title>Car Price Predictor</title>
    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }
        body {
            font-family: Arial, sans-serif;
            background: #f0f5f0;
            min-height: 100vh;
            display: flex;
            justify-content: center;
            align-items: center;
        }
        .container {
            background: #ffffff;
            padding: 40px;
            border-radius: 16px;
            width: 450px;
            box-shadow: 0 8px 30px rgba(0, 0, 0, 0.1);
            border-top: 4px solid #2e7d32;
        }
        h1 {
            text-align: center;
            margin-bottom: 25px;
            font-size: 24px;
            color: #2e7d32;
        }
        label {
            display: block;
            margin-top: 12px;
            font-size: 14px;
            color: #555;
            font-weight: bold;
        }
        input, select {
            width: 100%;
            padding: 10px;
            margin-top: 5px;
            border-radius: 8px;
            border: 1.5px solid #ccc;
            background: #fafffe;
            color: #333;
            font-size: 14px;
            transition: 0.3s;
        }
        input:focus, select:focus {
            outline: none;
            border-color: #43a047;
            box-shadow: 0 0 0 3px rgba(67, 160, 71, 0.15);
        }
        input::placeholder {
            color: #aaa;
        }
        .btn {
            width: 100%;
            padding: 12px;
            margin-top: 22px;
            border: none;
            border-radius: 8px;
            background: #2e7d32;
            color: white;
            font-size: 16px;
            cursor: pointer;
            transition: 0.3s;
        }
        .btn:hover {
            background: #1b5e20;
        }
        .result {
            text-align: center;
            margin-top: 20px;
            padding: 15px;
            background: #e8f5e9;
            border: 1px solid #a5d6a7;
            border-radius: 8px;
            font-size: 18px;
            font-weight: bold;
            color: #2e7d32;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>🚗 Car Price Predictor</h1>
        <form action="/predict" method="POST">

            <label>Present Price (in Lakhs)</label>
            <input type="text" name="present_price" placeholder="eg. 6.5" required>

            <label>Kilometers Driven</label>
            <input type="text" name="kms_driven" placeholder="eg. 45000" required>

            <label>Fuel Type</label>
            <select name="fuel_type">
                <option value="0">Petrol</option>
                <option value="1">Diesel</option>
                <option value="2">CNG</option>
            </select>

            <label>Seller Type</label>
            <select name="seller_type">
                <option value="0">Dealer</option>
                <option value="1">Individual</option>
            </select>

            <label>Transmission</label>
            <select name="transmission">
                <option value="0">Manual</option>
                <option value="1">Automatic</option>
            </select>

            <label>Number of Previous Owners</label>
            <input type="text" name="owner" placeholder="eg. 0" required>

            <label>Car Age (in years)</label>
            <input type="text" name="car_age" placeholder="eg. 5" required>

            <button type="submit" class="btn">Predict Price</button>
        </form>

        {% if prediction_text %}
        <div class="result">{{ prediction_text }}</div>
        {% endif %}
    </div>
</body>
</html>
'''

# Create the 'templates' directory if it doesn't exist
os.makedirs('templates', exist_ok=True)

with open('templates/index.html', 'w') as f:
    f.write(html_content)

print("Created index.html")

Created index.html


In [ ]:
# running the flask server
# click the link below once it starts, to stop it press the stop button in jupyter
!python app.py

 * Serving Flask app 'app'
 * Debug mode: on
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (inotify)
Traceback (most recent call last):
  File "/content/app.py", line 2, in <module>
    from flask import Flask, render_template, request
  File "/usr/local/lib/python3.12/dist-packages/flask/__init__.py", line 6, in <module>
    from .app import Flask as Flask
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 44, in <module>
    from .sansio.app import App
  File "/usr/local/lib/python3.12/dist-packages/flask/sansio/app.py", line 29, in <module>
    from ..templating import DispatchingJinjaLoader
  File "/usr/local/lib/python3.12/dist-packages/flask/templating.py", line 5, in <module>
    from jinja2 import BaseLoader
  File "/usr/local/lib/python3.12/dist-packages/jinja2/__init__.py", line 6, in <module>
    from .bccache import BytecodeCache as BytecodeCache
  File "/usr/local/lib/python3.12/dist-packages/jinja2/bccache.py", line 